# Geertsma–Van Opstal Reservoir Subsidence

This notebook demonstrates the full spatial workflow using synthetic reservoir data:

**pressure depletion → reservoir compaction → influence function → surface subsidence → visualization**

## Units and sign convention

- coordinates and depth: m
- pressure depletion: bar
- compaction coefficient: 1/bar
- compaction volume: m³
- calculated subsidence: m
- plotted subsidence: mm

Subsidence is positive downward.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from reservoir_subsidence.compaction import calculate_compaction_volume
from reservoir_subsidence.grid import make_surface_grid, results_dataframe
from reservoir_subsidence.van_opstal import calculate_subsidence
from reservoir_subsidence.visualization import (
    plot_subsidence_map,
    plot_center_profile,
)

## Load synthetic reservoir cells

In [ ]:
data = pd.read_csv(ROOT / "data" / "synthetic_reservoir.csv")
data.head()

## Reservoir compaction

```math
\Delta V = A h C_m \Delta p
```

In [ ]:
data["compaction_volume_m3"] = calculate_compaction_volume(
    data["source_cell_area_m2"],
    data["thickness_m"],
    data["compaction_coefficient_per_bar"],
    data["pressure_depletion_bar"],
)

data["basement_depth_m"] = (
    data["reservoir_depth_m"] * data["basement_depth_factor"]
)

data.groupby("reservoir_id")["compaction_volume_m3"].sum()

## Surface grid

In [ ]:
x, y, xx, yy = make_surface_grid(
    xmin_m=-4500,
    xmax_m=6500,
    ymin_m=-4000,
    ymax_m=4500,
    spacing_m=200,
)

print(f"Grid points: {xx.size:,}")

## Calculate each reservoir contribution

In [ ]:
reservoir_results = {}

for reservoir_id, group in data.groupby("reservoir_id"):
    reservoir_results[reservoir_id] = calculate_subsidence(
        xx,
        yy,
        group["x_m"].to_numpy(),
        group["y_m"].to_numpy(),
        group["reservoir_depth_m"].to_numpy(),
        group["basement_depth_m"].to_numpy(),
        group["compaction_volume_m3"].to_numpy(),
    )

total_subsidence_m = sum(reservoir_results.values())
print(f"Maximum subsidence: {total_subsidence_m.max()*1000:.2f} mm")

## Total subsidence map

In [ ]:
fig, ax = plot_subsidence_map(
    xx,
    yy,
    total_subsidence_m,
    output_path=ROOT / "figures" / "total_subsidence_map.png",
)
plt.show()

## Centre-line profile

In [ ]:
fig, ax = plot_center_profile(
    x,
    total_subsidence_m,
    output_path=ROOT / "figures" / "subsidence_center_profile.png",
)
plt.show()

## Engineering interpretation

The total surface response is the superposition of the subsidence bowls generated by the two synthetic depleted reservoirs.

Bowl amplitude and width are controlled by the imposed compaction field, reservoir depth and geometry, and the assumed rigid-basement depth.